In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegressionCV, LassoCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

from utils.data_loader import load_data

# Loading data

In [2]:
final_clean_strained = load_data(db_level=0)

In [3]:
peak_curvs = final_clean_strained["PeakI Curvature"].apply(pd.Series)
peak_curvs.columns = [
    "PeakI Left Curvature",
    "PeakI Overall Curvature",
    "PeakI Right Curvature",
]
trough_curvs = final_clean_strained["TroughI Curvature"].apply(pd.Series)
trough_curvs.columns = [
    "TroughI Left Curvature",
    "TroughI Overall Curvature",
    "TroughI Right Curvature",
]
final_clean_strained = pd.concat(
    [final_clean_strained, peak_curvs, trough_curvs], axis=1
)

In [4]:
def split_by_mouse(
    data,
    split_on="Subject",
    stratify_on="Group",
    test_size=0.2,
    val_size=0.18,
    random_state=22,
    return_idx=False,
):
    mice = data[[split_on, stratify_on]].drop_duplicates().set_index(split_on)

    train, test = train_test_split(
        mice.index,
        test_size=test_size,
        shuffle=True,
        stratify=mice[stratify_on],
        random_state=random_state,
    )

    train_indices = data[split_on].isin(train)
    test_indices = data[split_on].isin(test)

    train2, val = train_test_split(
        train,
        test_size=val_size,
        shuffle=True,
        stratify=mice.loc[train]["Group"],
        random_state=random_state,
    )

    train2_indices = data[split_on].isin(train2)
    val_indices = data[split_on].isin(val)

    data["DataGroup"] = ""
    data.loc[train2_indices, "DataGroup"] = "Train"
    data.loc[val_indices, "DataGroup"] = "Validate"
    data.loc[test_indices, "DataGroup"] = "Test"
    if return_idx:
        return data, train_indices, train2_indices, val_indices, test_indices
    return data

# Train test split

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    FunctionTransformer,
    MinMaxScaler,
)


def RF_data_prep(
    data,
    num_features,
    cat_features,
    log_features,
    minmax_features,
    target="SynapsesPerIHC",
):
    X = data[num_features + cat_features + log_features + minmax_features]
    y = data[target]

    X_train, X_test, y_train, y_test = (
        X[data.DataGroup != "Test"],
        X[data.DataGroup == "Test"],
        y[data.DataGroup != "Test"],
        y[data.DataGroup == "Test"],
    )

    log_scale_pipeline = Pipeline(
        [
            ("log_transform", FunctionTransformer(np.log1p, validate=True)),
            ("scaler", StandardScaler()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_features),
            ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_features),
            ("log_transform", log_scale_pipeline, log_features),
            ("minmax_scale", MinMaxScaler(), minmax_features),
        ]
    )

    return X_train, X_test, y_train, y_test, preprocessor

In [6]:
final_clean_strained = split_by_mouse(final_clean_strained)
num_features = ["Amplitude", "Slope", "Distance", "Level(dB)"]
cat_features = ["vx", "Strain (binary)"]
log_features = [
    "Frequency(kHz)",
    "Total Variance",
    "PeakI Left Curvature",
    "PeakI Overall Curvature",
    "PeakI Right Curvature",
    "TroughI Left Curvature",
    "TroughI Overall Curvature",
    "TroughI Right Curvature",
]
minmax_features = ["Noise", "Time (hrs)"]
X_train, X_test, y_train, y_test, preprocessor = RF_data_prep(
    final_clean_strained, num_features, cat_features, log_features, minmax_features
)

In [7]:
final_clean_strained.columns = [
    "Subject",
    "Frequency",
    "Level",
    "Amplitude",
    "vx",
    "SynapsesPerIHC",
    "IHCs",
    "OrphansPerIHC",
    "Waveform",
    "WaveI",
    "Peaks",
    "Troughs",
    "Slope",
    "TotalVariance",
    "Distance",
    "PeakICurvature",
    "TroughICurvature",
    "Strain",
    "Sex",
    "Group",
    "Noise",
    "Time(str)",
    "Time_hrs",
    "Strain_b",
    "PeakILeftCurvature",
    "PeakIOverallCurvature",
    "PeakIRightCurvature",
    "TroughILeftCurvature",
    "TroughIOverallCurvature",
    "TroughIRightCurvature",
    "DataGroup",
]

In [8]:
cols = [
    "Amplitude",
    "Slope",
    "Distance",
    "TotalVariance",
    "PeakILeftCurvature",
    "PeakIOverallCurvature",
    "PeakIRightCurvature",
    "TroughILeftCurvature",
    "TroughIOverallCurvature",
    "TroughIRightCurvature",
]

reformatted = []

for col in cols:
    df = (
        final_clean_strained.pivot_table(
            index=["Subject", "Frequency"], columns="Level", values=col
        )
        .rename_axis(columns="Level_dB")
        .add_prefix(f"{col}_")
    )
    reformatted.append(df)
reformatted = pd.concat(reformatted, axis=1)

# Add Noise column
noise = final_clean_strained.groupby(["Subject", "Frequency"])["Noise"].mean()
synapses = final_clean_strained.groupby(["Subject", "Frequency"])[
    "SynapsesPerIHC"
].mean()
groups = final_clean_strained.groupby(["Subject", "Frequency"])[
    ["DataGroup", "Strain_b"]
].first()

# Combine
reformatted = reformatted.join(noise).join(synapses).join(groups)

# fill NaNs with interpolated/extrapolated values:
for col in cols:
    amp_cols = [c for c in reformatted.columns if c.startswith(col)]
    reformatted[amp_cols] = reformatted[amp_cols].interpolate(
        axis=1, limit_direction="both"
    )
reformatted.reset_index(inplace=True)
reformatted.head(10)

,Subject,Frequency,Amplitude_10.0,Amplitude_15.0,Amplitude_20.0,Amplitude_25.0,Amplitude_30.0,Amplitude_35.0,Amplitude_40.0,Amplitude_45.0,...,TroughIRightCurvature_55.0,TroughIRightCurvature_60.0,TroughIRightCurvature_65.0,TroughIRightCurvature_70.0,TroughIRightCurvature_75.0,TroughIRightCurvature_80.0,Noise,SynapsesPerIHC,DataGroup,Strain_b
0,WPZ100,8.0,0.077961,0.077961,0.029777,0.090885,0.137358,0.296385,0.381854,0.519497,...,14.796865,25.079451,13.892941,2.706432,1.739748,0.773064,0.00,15.892142,Test,1
1,WPZ100,11.3,0.050718,0.050718,0.000000,0.168245,0.302752,0.392846,0.477460,0.668926,...,6.280679,5.094698,2.653182,0.211666,0.260892,0.310118,0.00,16.492360,Test,1
2,WPZ100,16.0,0.059556,0.077295,0.047106,0.121329,0.202553,0.426837,0.424367,0.526245,...,3.128261,0.390464,0.450276,0.510088,0.333558,0.157028,0.00,17.564103,Test,1
3,WPZ100,22.6,0.063072,0.063072,0.060886,0.098964,0.223176,0.255710,0.359520,0.548261,...,3.570113,1.643706,0.862335,0.080963,0.440617,0.800270,0.00,17.381424,Test,1
4,WPZ100,32.0,0.091900,0.091900,0.257800,0.321347,0.333101,0.498574,0.664046,0.841320,...,0.821376,0.622318,2.515715,4.409113,2.254731,0.100350,0.00,18.314239,Test,1
5,WPZ100,45.2,0.079988,0.079988,0.064308,0.129407,0.054423,0.130737,0.151740,0.219944,...,6.313868,12.019363,6.413762,0.808162,0.566801,0.325440,0.00,17.846405,Test,1
6,WPZ101,32.0,0.054107,0.087369,0.057180,0.286407,0.411283,0.572701,0.734120,1.078561,...,11.750996,22.945029,13.590213,4.235397,42.735342,81.235287,0.00,16.193497,Train,1
7,WPZ101,45.2,0.043558,0.111223,0.131910,0.057465,0.140146,0.098869,0.097000,0.115595,...,18.925156,4.778665,10.090546,6.866563,17.967283,8.283850,0.00,15.351421,Train,1
8,WPZ102,8.0,0.067254,0.180315,0.126208,0.006525,0.139418,0.266449,0.441124,0.574855,...,4.034692,3.181216,14.842119,26.503023,13.575201,0.647380,0.94,14.334043,Train,1
9,WPZ102,11.3,0.195773,0.029398,0.095163,0.233439,0.310324,0.502138,0.693951,0.728227,...,3.145657,2.266414,1.203518,0.140623,0.097414,0.054206,0.94,16.569307,Train,1


# Predicting Binary Noise Groups (<91dB vs. >= 91dB)

In [9]:
reformatted["Noise"].value_counts()

Noise
0.00    193
0.98    115
0.94    100
0.90     82
0.94     37
0.98     28
1.01     16
0.98     14
0.94     13
0.90      9
0.90      9
Name: count, dtype: int64

In [10]:
num_features = []
cat_features = []
log_features = ["Frequency"]
for col in [
    "TotalVariance",
    "PeakILeftCurvature",
    "PeakIOverallCurvature",
    "PeakIRightCurvature",
    "TroughILeftCurvature",
    "TroughIOverallCurvature",
    "TroughIRightCurvature",
]:
    cols = [c for c in reformatted.columns if c.startswith(col)]
    log_features.extend(cols)

for col in ["Amplitude", "Slope", "Distance"]:
    cols = [c for c in reformatted.columns if c.startswith(col)]
    num_features.extend(cols)

reformatted["target_90"] = [0 if x else 1 for x in (reformatted["Noise"] <= 0.91)]
X = reformatted[num_features + cat_features + log_features]
y = reformatted["target_90"]

X_train, X_test, y_train, y_test = (
    X[reformatted.DataGroup != "Test"],
    X[reformatted.DataGroup == "Test"],
    y[reformatted.DataGroup != "Test"],
    y[reformatted.DataGroup == "Test"],
)

log_scale_pipeline = Pipeline(
    [
        ("log_transform", FunctionTransformer(np.log1p, validate=True)),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_features),
        ("log_transform", log_scale_pipeline, log_features),
    ]
)

In [11]:
rf_params = {
    "rf__n_estimators": [25, 50, 100, 150, 200, 30, 400, 500],
    "rf__max_depth": [5, 10, 20, 30, 40, 50, None],
    "rf__min_samples_leaf": [2, 5, 10, 15, 20, 30],
    "rf__min_samples_split": [2, 5, 10, 15, 20, 30],
    "rf__max_features": ["sqrt", "log2", 0.3, 0.5],
}

rf_pipeline = Pipeline(
    [("preprocessor", preprocessor), ("rf", RandomForestClassifier(random_state=1))]
)

# Use RandomizedSearchCV for efficiency
rf_search = RandomizedSearchCV(
    rf_pipeline,
    rf_params,
    cv=5,
    random_state=1,
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)
r2 = rf_search.score(X_test, y_test)
best_rf_model = rf_search.best_estimator_

In [12]:
X = reformatted[num_features + cat_features + log_features]
rf_pipeline.fit(X_train, y_train)
reformatted["noise_preds"] = rf_pipeline.predict(X)

In [13]:
reformatted["noise_proba"] = rf_pipeline.predict_proba(X)[:, 1]

In [14]:
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV


def make_comparison_plot(name, X_train, y_train, X_test, y_test, clf):
    fig, ax = plt.subplots(figsize=(7, 6))
    preds = clf.predict_proba(X_test)[:, 1]
    CalibrationDisplay.from_predictions(
        y_test, preds, n_bins=10, ax=ax, name="Uncalibrated"
    )
    cal_clf = CalibratedClassifierCV(clf, method=name, n_jobs=-1)
    cal_clf.fit(X_train, y_train)
    cal_preds = cal_clf.predict_proba(X_test)[:, 1]
    CalibrationDisplay.from_predictions(
        y_test, cal_preds, n_bins=10, ax=ax, name="Calibrated"
    )
    plt.title(f"Uncalibrated probabilities vs. calibrated probabilities ({name})")
    plt.show()

In [15]:
cal_clf = CalibratedClassifierCV(rf_pipeline, method="isotonic", n_jobs=-1)
cal_clf.fit(X_train, y_train)
reformatted["calibrated_proba"] = cal_clf.predict_proba(X)[:, 1]

In [16]:
preds = reformatted.groupby(["Subject"]).agg({"calibrated_proba": "mean"})
preds["noise_proba"] = np.where(preds > 0.5, 1, 0)
preds_dict = preds["noise_proba"].to_dict()
reformatted["noise_preds"] = reformatted["Subject"].replace(preds_dict)
train = reformatted[reformatted.DataGroup != "Test"]
test = reformatted[reformatted.DataGroup == "Test"]
print("Train Acc:", (train["noise_preds"] == train["target_90"]).mean())
print("Test Acc:", (test["noise_preds"] == test["target_90"]).mean())
print("Overall Acc:", (reformatted["noise_preds"] == reformatted["target_90"]).mean())

Train Acc: 1.0
Test Acc: 0.904
Overall Acc: 0.9805194805194806
